# ⚙️ 02 — Image Preprocessing & Augmentation
**Project:** Diabetic Retinopathy Detection

**Goal:** Implement Ben Graham's preprocessing technique, build the Albumentations augmentation pipeline, create stratified train/val/test splits, and verify the final `tf.data` pipeline.

---

## 📦 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import tensorflow as tf

DATA_DIR  = Path('../data/raw')
PROC_DIR  = Path('../data/processed')
PROC_DIR.mkdir(exist_ok=True)

TRAIN_DIR = DATA_DIR / 'train_images'
IMG_SIZE  = 224
BATCH_SIZE = 32
SEED = 42

LABEL_MAP = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 🔬 2. Ben Graham Preprocessing

This technique subtracts a heavily blurred version of the image from itself, which:
- Removes uneven illumination from the camera/fundus setup
- Enhances microaneurysms, exudates, and hemorrhages
- Applies a circular crop to remove black borders

In [ ]:
def crop_image_from_gray(img, tol=7):
    """Remove dark circular borders from fundus images."""
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray > tol
        check_shape = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_shape == 0:
            return img
        return img[np.ix_(mask.any(1), mask.any(0))]


def ben_graham_preprocess(img, sigmaX=10):
    """
    Ben Graham preprocessing for retinal fundus images.
    Source: Kaggle 2015 DR Competition - Ben Graham (1st place)
    """
    img = crop_image_from_gray(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    # Subtract blurred version to enhance local structure
    img = cv2.addWeighted(img, 4,
                          cv2.GaussianBlur(img, (0, 0), sigmaX), -4,
                          128)
    # Apply circular mask
    b = np.zeros(img.shape, dtype=np.uint8)
    cv2.circle(b, (IMG_SIZE // 2, IMG_SIZE // 2),
               int(IMG_SIZE * 0.9 // 2), (1, 1, 1), -1, 8, 0)
    img = img * b + 128 * (1 - b)
    return img.astype(np.uint8)


print('Preprocessing function defined.')

### Visualize: Raw vs. Preprocessed

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
df['image_path'] = df['id_code'].apply(lambda x: str(TRAIN_DIR / f'{x}.png'))

sample_paths = [df[df['diagnosis'] == g].sample(1, random_state=42)['image_path'].values[0] for g in range(5)]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (path, grade) in enumerate(zip(sample_paths, range(5))):
    raw = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    proc = ben_graham_preprocess(raw.copy())
    raw_resized = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
    axes[0, i].imshow(raw_resized)
    axes[0, i].set_title(f'Grade {grade}: {LABEL_MAP[grade]}\n(Raw)', fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(proc)
    axes[1, i].set_title(f'Grade {grade}: {LABEL_MAP[grade]}\n(Ben Graham)', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Raw vs. Ben Graham Preprocessed Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔁 3. Augmentation Pipeline

In [ ]:
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GridDistortion(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Visualize augmentations
sample_img = cv2.cvtColor(cv2.imread(sample_paths[2]), cv2.COLOR_BGR2RGB)
sample_img = ben_graham_preprocess(sample_img)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes[0, 0].imshow(sample_img); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
for i in range(1, 10):
    aug = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.9), A.RandomBrightnessContrast(p=0.7),
    ])(image=sample_img)['image']
    r, c = divmod(i, 5)
    axes[r, c].imshow(aug); axes[r, c].set_title(f'Augmented #{i}'); axes[r, c].axis('off')

plt.suptitle('Augmentation Examples (Grade 2 — Moderate DR)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## ✂️ 4. Stratified Train / Val / Test Split

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['diagnosis'], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['diagnosis'], random_state=SEED)

train_df.to_csv(PROC_DIR / 'train.csv', index=False)
val_df.to_csv(PROC_DIR   / 'val.csv',   index=False)
test_df.to_csv(PROC_DIR  / 'test.csv',  index=False)

print(f'Train : {len(train_df):,} images ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):,}  images ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):,}  images ({len(test_df)/len(df)*100:.1f}%)')

# Verify class proportions are maintained
print('\nClass distribution preserved (proportions):')
comparison = pd.DataFrame({
    'Full':  df['diagnosis'].value_counts(normalize=True).sort_index(),
    'Train': train_df['diagnosis'].value_counts(normalize=True).sort_index(),
    'Val':   val_df['diagnosis'].value_counts(normalize=True).sort_index(),
    'Test':  test_df['diagnosis'].value_counts(normalize=True).sort_index(),
}).round(3)
print(comparison)

## ⚙️ 5. Compute Class Weights for Loss

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3, 4])
class_weights = compute_class_weight('balanced', classes=classes, y=train_df['diagnosis'].values)
class_weight_dict = dict(zip(classes, class_weights))

print('Class weights for training:')
for grade, weight in class_weight_dict.items():
    print(f'  Grade {grade} ({LABEL_MAP[grade]:15s}): {weight:.3f}')

## ✅ 6. Summary

| Step | Output |
|------|--------|
| Ben Graham preprocessing | Enhances lesion visibility, removes illumination artifacts |
| Augmentation pipeline | 8 transforms — flips, rotation, color jitter, blur, distortion |
| Data split | 80% train / 10% val / 10% test — stratified |
| Class weights | Computed and ready to pass to `model.fit()` |

**Next step:** `03_Model_Training.ipynb` — build and train EfficientNetB3 with two-phase fine-tuning.